## Ide kezdem el a Binance letöltését

In [ ]:
from __future__ import annotations

import io
import zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import Protocol

import requests

In [ ]:
@dataclass(frozen=True)
class MarketDataRequest:
    symbol: str
    interval: str
    year: int
    month: int


@dataclass(frozen=True)
class DownloadResult:
    provider: str
    url: str
    files: list[Path]


class MarketDataProvider(Protocol):
    name: str

    def build_url(self, request: MarketDataRequest) -> str:
        ...

    def download(self, request: MarketDataRequest, output_dir: Path) -> DownloadResult:
        ...


In [ ]:
class BinanceProvider:
    name = "binance"
    base_url = "https://data.binance.vision"

    def __init__(self, timeout_sec: int = 30):
        self.timeout_sec = timeout_sec
        self.session = requests.Session()

    def build_url(self, request: MarketDataRequest) -> str:
        month = f"{request.month:02d}"

        return (
            f"{self.base_url}/data/spot/monthly/klines/"
            f"{request.symbol}/{request.interval}/"
            f"{request.symbol}-{request.interval}-{request.year}-{month}.zip"
        )

    def download(self, request: MarketDataRequest, output_dir: Path) -> DownloadResult:
        url = self.build_url(request)

        response = self.session.get(url, timeout=self.timeout_sec)

        if response.status_code == 404:
            raise FileNotFoundError(f"Binance file not found: {url}")

        if response.status_code != 200:
            raise RuntimeError(f"Binance HTTP error {response.status_code}: {url}")

        files = self._extract_csv_files(
            zip_bytes=response.content,
            output_dir=output_dir,
        )

        return DownloadResult(
            provider=self.name,
            url=url,
            files=files,
        )

    def _extract_csv_files(self, zip_bytes: bytes, output_dir: Path) -> list[Path]:
        output_dir.mkdir(parents=True, exist_ok=True)

        extracted_files: list[Path] = []

        with zipfile.ZipFile(io.BytesIO(zip_bytes)) as archive:
            for member in archive.infolist():
                if not member.filename.lower().endswith(".csv"):
                    continue

                member_path = Path(member.filename)

                if member_path.is_absolute() or ".." in member_path.parts:
                    raise ValueError(f"Unsafe ZIP entry: {member.filename}")

                archive.extract(member, output_dir)
                extracted_files.append(output_dir / member.filename)

        if not extracted_files:
            raise ValueError("No CSV file found in Binance ZIP.")

        return extracted_files


In [ ]:
provider = BinanceProvider()

request = MarketDataRequest(
    symbol="BTCUSDT",
    interval="1m",
    year=2024,
    month=1,
)

result = provider.download(
    request=request,
    output_dir=Path("data/binance"),
)

result


In [ ]:
from pathlib import Path

from providers.binance import BinanceProvider, BinanceRequest


provider = BinanceProvider()

request = BinanceRequest(
    symbol="BTCUSDT",
    interval="1m",
    year=2024,
    month=1,
)

result = provider.download(
    request=request,
    output_dir=Path("data/raw/binance"),
)

result


In [ ]:
from pathlib import Path
import csv
from datetime import datetime, timezone


csv_path = result.files[0]

expected_column_count = 12

row_count = 0
bad_column_count = 0
bad_timestamp_count = 0
bad_number_count = 0
duplicate_timestamp_count = 0

timestamps = set()
min_timestamp = None
max_timestamp = None
previous_timestamp = None
is_time_ordered = True

number_columns = [1, 2, 3, 4, 5, 7, 8, 9, 10]

with open(csv_path, "r", encoding="utf-8") as file:
    reader = csv.reader(file)

    for row in reader:
        row_count += 1

        if len(row) != expected_column_count:
            bad_column_count += 1
            continue

        try:
            timestamp = int(row[0])
            # timestamp_dt = datetime.fromtimestamp(timestamp / 1000)
            timestamp_dt = datetime.fromtimestamp(timestamp / 1000, tz=timezone.utc)
        except ValueError:
            bad_timestamp_count += 1
            continue

        if timestamp in timestamps:
            duplicate_timestamp_count += 1
        else:
            timestamps.add(timestamp)

        if previous_timestamp is not None and timestamp < previous_timestamp:
            is_time_ordered = False

        previous_timestamp = timestamp

        if min_timestamp is None or timestamp_dt < min_timestamp:
            min_timestamp = timestamp_dt

        if max_timestamp is None or timestamp_dt > max_timestamp:
            max_timestamp = timestamp_dt

        for col_index in number_columns:
            try:
                float(row[col_index])
            except ValueError:
                bad_number_count += 1


validation_report = {
    "file": str(csv_path),
    "row_count": row_count,
    "bad_column_count": bad_column_count,
    "bad_timestamp_count": bad_timestamp_count,
    "bad_number_count": bad_number_count,
    "duplicate_timestamp_count": duplicate_timestamp_count,
    "is_time_ordered": is_time_ordered,
    "min_timestamp": min_timestamp,
    "max_timestamp": max_timestamp,
}

validation_report


In [ ]:
from pathlib import Path

from providers.binance import BinanceProvider, BinanceRequest
from validators.binance_csv import validate_binance_csv


provider = BinanceProvider()

request = BinanceRequest(
    symbol="BTCUSDT",
    interval="1m",
    year=2024,
    month=1,
)

download_result = provider.download(
    request=request,
    output_dir=Path("data/raw/binance"),
)

validation_result = validate_binance_csv(
    csv_path=download_result.files[0],
)

download_result, validation_result


In [ ]:
from transformers.binance_ohlcv import transform_binance_csv_to_ohlcv


ohlcv_df = transform_binance_csv_to_ohlcv(
    csv_path=download_result.files[0],
)

ohlcv_df.head()


In [ ]:
from pathlib import Path

from writers.ohlcv_parquet import write_ohlcv_to_parquet


parquet_path = write_ohlcv_to_parquet(
    df=ohlcv_df,
    output_path=Path("data/bronze/binance/btcusd/2024/01/BTCUSDT-1m-2024-01.parquet"),
)

parquet_path


In [ ]:
import pandas as pd


parquet_df = pd.read_parquet(parquet_path)

parquet_df.head()


In [ ]:
from pathlib import Path

from metadata.manifest import build_manifest, write_manifest


manifest = build_manifest(
    provider="binance",
    symbol="BTCUSDT",
    interval="1m",
    year=2024,
    month=1,
    source_url=download_result.url,
    raw_file_path=download_result.files[0],
    parquet_file_path=parquet_path,
    validation_result=validation_result,
    ohlcv_df=ohlcv_df,
)

manifest_path = write_manifest(
    manifest=manifest,
    output_path=Path("data/bronze/binance/btcusd/2024/01/_MANIFEST.json"),
)

manifest_path


In [ ]:
import json

with open(manifest_path, "r", encoding="utf-8") as file:
    loaded_manifest = json.load(file)

loaded_manifest


In [ ]:
from metadata.manifest import build_manifest, write_manifest
from metadata.success_marker import write_success_marker


In [ ]:
from pathlib import Path

from metadata.success_marker import write_success_marker


bronze_dir = Path("data/bronze/binance/btcusd/2024/01")

success_path = write_success_marker(
    output_path=bronze_dir / "_SUCCESS",
)

success_path


In [ ]:
from pathlib import Path

from uploaders.azure_blob import init_azure_client


blob_service_client, container_name = init_azure_client(
    env_path=Path(".env"),
)

container_name


In [ ]:
from pathlib import Path


parquet_path = Path(
    r"D:\Egyetem\On_lab_V2\data\bronze\binance\btcusd\2024\01\BTCUSDT-1m-2024-01.parquet"
)

parquet_path.exists()


In [ ]:
from uploaders.azure_blob import upload_file


parquet_blob_name = "bronze/binance/btcusd/2024/01/BTCUSDT-1m-2024-01.parquet"

upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=parquet_path,
    blob_name=parquet_blob_name,
)


In [ ]:
blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob=parquet_blob_name,
)

blob_client.exists()



In [ ]:
manifest_path = Path(
    r"D:\Egyetem\On_lab_V2\data\bronze\binance\btcusd\2024\01\_MANIFEST.json"
)

manifest_blob_name = "bronze/binance/btcusd/2024/01/_MANIFEST.json"

upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=manifest_path,
    blob_name=manifest_blob_name,
)


In [ ]:
blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob=manifest_blob_name,
)

blob_client.exists()


In [ ]:
success_path = Path(
    r"D:\Egyetem\On_lab_V2\data\bronze\binance\btcusd\2024\01\_SUCCESS"
)

success_blob_name = "bronze/binance/btcusd/2024/01/_SUCCESS"

upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=success_path,
    blob_name=success_blob_name,
)


In [ ]:
blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob=success_blob_name,
)

blob_client.exists()


In [ ]:
raw_csv_path = Path(
    r"D:\Egyetem\On_lab_V2\data\raw\binance\BTCUSDT-1m-2024-01.csv"
)

raw_csv_blob_name = "raw/binance/btcusd/2024/01/BTCUSDT-1m-2024-01.csv"

upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=raw_csv_path,
    blob_name=raw_csv_blob_name,
)


In [ ]:
blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob=raw_csv_blob_name,
)

blob_client.exists()


In [ ]:
import importlib
import uploaders.azure_blob

importlib.reload(uploaders.azure_blob)


In [ ]:
from uploaders.azure_blob import blob_exists


success_blob_name = "bronze/binance/btcusd/2024/01/_SUCCESS"

blob_exists(
    blob_service_client=blob_service_client,
    container_name=container_name,
    blob_name=success_blob_name,
)


In [ ]:
from datetime import date

from planner.time_windows import generate_monthly_windows, generate_daily_windows


monthly_windows = generate_monthly_windows(
    start_date=date(2024, 1, 1),
    end_date=date(2024, 3, 31),
)

daily_windows = generate_daily_windows(
    start_date=date(2024, 1, 1),
    end_date=date(2024, 1, 5),
)

monthly_windows, daily_windows


In [ ]:
from pathlib import Path

from config_loader.csv_config import (
    load_broker_strategy,
    load_broker_asset_matrix,
    load_download_period,
)


config_dir = Path("config")

broker_strategy_df = load_broker_strategy(config_dir)
broker_asset_matrix_df = load_broker_asset_matrix(config_dir)
download_period_df = load_download_period(config_dir)

broker_strategy_df, broker_asset_matrix_df, download_period_df


In [ ]:
from planner.download_plan import build_download_plan


plan = build_download_plan(
    broker_strategy_df=broker_strategy_df,
    broker_asset_matrix_df=broker_asset_matrix_df,
    download_period_df=download_period_df,
)

len(plan), plan[:3]


In [ ]:
from paths.data_paths import build_data_paths


item = plan[0]

paths = build_data_paths(
    item=item,
    interval="1m",
)

paths


In [ ]:
from pathlib import Path

from providers.binance import BinanceProvider, BinanceRequest
from validators.binance_csv import validate_binance_csv
from transformers.binance_ohlcv import transform_binance_csv_to_ohlcv
from writers.ohlcv_parquet import write_ohlcv_to_parquet
from metadata.manifest import build_manifest, write_manifest
from metadata.success_marker import write_success_marker
from uploaders.azure_blob import upload_file, blob_exists
from paths.data_paths import build_data_paths


In [ ]:
item = plan[0]

interval = "1m"

paths = build_data_paths(
    item=item,
    interval=interval,
)

paths


In [ ]:
already_done = blob_exists(
    blob_service_client=blob_service_client,
    container_name=container_name,
    blob_name=paths.success_blob_name,
)

already_done


In [ ]:
provider = BinanceProvider()

request = BinanceRequest(
    symbol=item.broker_symbol,
    interval=interval,
    year=item.window.start_date.year,
    month=item.window.start_date.month,
)

download_result = provider.download(
    request=request,
    output_dir=paths.local_raw_file.parent,
)

download_result


In [ ]:
validation_result = validate_binance_csv(
    csv_path=download_result.files[0],
)

validation_result


In [ ]:
ohlcv_df = transform_binance_csv_to_ohlcv(
    csv_path=download_result.files[0],
)

ohlcv_df.head()


In [ ]:
from pathlib import Path

from uploaders.azure_blob import init_azure_client


blob_service_client, container_name = init_azure_client(
    env_path=Path(".env"),
)

container_name

In [ ]:
manifest = build_manifest(
    provider=item.broker,
    symbol=item.broker_symbol,
    interval=interval,
    year=item.window.start_date.year,
    month=item.window.start_date.month,
    source_url=download_result.url,
    raw_file_path=download_result.files[0],
    parquet_file_path=parquet_path,
    validation_result=validation_result,
    ohlcv_df=ohlcv_df,
)

manifest_path = write_manifest(
    manifest=manifest,
    output_path=paths.local_manifest_file,
)

manifest_path


In [ ]:
success_path = write_success_marker(
    output_path=paths.local_success_file,
)

success_path


In [ ]:
success_path.exists(), success_path.stat().st_size


In [ ]:
upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=download_result.files[0],
    blob_name=paths.raw_blob_name,
)

upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=paths.local_bronze_parquet_file,
    blob_name=paths.bronze_parquet_blob_name,
)

upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=paths.local_manifest_file,
    blob_name=paths.manifest_blob_name,
)

upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=paths.local_success_file,
    blob_name=paths.success_blob_name,
)


In [ ]:
blob_exists(
    blob_service_client=blob_service_client,
    container_name=container_name,
    blob_name=paths.success_blob_name,
)


In [ ]:
from pipelines.binance_pipeline import run_binance_plan_item


result = run_binance_plan_item(
    item=plan[0],
    interval="1m",
    blob_service_client=blob_service_client,
    container_name=container_name,
)

result


In [ ]:
from pathlib import Path

from config_loader.csv_config import (
    load_broker_strategy,
    load_broker_asset_matrix,
    load_download_period,
)
from planner.download_plan import build_download_plan
from uploaders.azure_blob import init_azure_client


config_dir = Path("config")

broker_strategy_df = load_broker_strategy(config_dir)
broker_asset_matrix_df = load_broker_asset_matrix(config_dir)
download_period_df = load_download_period(config_dir)

plan = build_download_plan(
    broker_strategy_df=broker_strategy_df,
    broker_asset_matrix_df=broker_asset_matrix_df,
    download_period_df=download_period_df,
)

blob_service_client, container_name = init_azure_client(
    env_path=Path(".env"),
)

len(plan), container_name


In [ ]:
result = run_binance_plan_item(
    item=plan[1],
    interval="1m",
    blob_service_client=blob_service_client,
    container_name=container_name,
)

result


In [ ]:
results = []

for item in plan[:3]:
    result = run_binance_plan_item(
        item=item,
        interval="1m",
        blob_service_client=blob_service_client,
        container_name=container_name,
    )

    results.append(result)

results


In [ ]:
from pipelines.binance_pipeline import run_binance_plan


results = run_binance_plan(
    plan=plan[:3],
    interval="1m",
    blob_service_client=blob_service_client,
    container_name=container_name,
)

results


In [ ]:
import importlib
import pipelines.binance_pipeline

importlib.reload(pipelines.binance_pipeline)

from pipelines.binance_pipeline import run_binance_plan


In [ ]:
results = run_binance_plan(
    plan=plan[:3],
    interval="1m",
    blob_service_client=blob_service_client,
    container_name=container_name,
)

results


In [ ]:
results = run_binance_plan(
    plan=plan[:3],
    interval="1m",
    blob_service_client=blob_service_client,
    container_name=container_name,
)

results

In [ ]:
import importlib
import pipelines.binance_runner

importlib.reload(pipelines.binance_runner)

from pipelines.binance_runner import run_binance_from_config


In [ ]:
results = run_binance_from_config(
    start_index=4,
    limit=1,
)

results


## Dukascopy elkezdése

In [ ]:
from pathlib import Path


output_dir = Path("data/staging/dukascopy/eurusd/2024/01")
output_dir.mkdir(parents=True, exist_ok=True)

bi5_path = output_dir / "EURUSD-2024-01-02-12_ticks.bi5"

bi5_path.write_bytes(response.content)

bi5_path


In [ ]:
bi5_path.exists(), bi5_path.stat().st_size


In [ ]:
import lzma
import struct


In [ ]:
import lzma


compressed = bi5_path.read_bytes()

decompressed = lzma.decompress(compressed)

len(compressed), len(decompressed)


In [ ]:
record_size = 20
record_count = len(decompressed) // record_size

len(decompressed), record_count, len(decompressed) % record_size


In [ ]:
import struct
from datetime import timedelta


price_scale = 100000
record_size = 20

records = []

for i in range(5):
    offset = i * record_size

    time_delta_ms, ask, bid, ask_volume, bid_volume = struct.unpack(
        ">iii ff",
        decompressed[offset : offset + record_size],
    )

    tick_time = ts + timedelta(milliseconds=time_delta_ms)

    records.append(
        {
            "timestamp": tick_time,
            "bid": bid / price_scale,
            "ask": ask / price_scale,
            "bid_volume": bid_volume,
            "ask_volume": ask_volume,
        }
    )

records


In [ ]:
import pandas as pd
import struct
from datetime import timedelta


price_scale = 100000
record_size = 20

rows = []

for offset in range(0, len(decompressed), record_size):
    time_delta_ms, ask, bid, ask_volume, bid_volume = struct.unpack(
        ">iii ff",
        decompressed[offset : offset + record_size],
    )

    rows.append(
        {
            "timestamp": ts + timedelta(milliseconds=time_delta_ms),
            "bid": bid / price_scale,
            "ask": ask / price_scale,
            "bid_volume": bid_volume,
            "ask_volume": ask_volume,
        }
    )

ticks_df = pd.DataFrame(rows)

ticks_df.head()


In [ ]:
ticks_df.shape, ticks_df["timestamp"].min(), ticks_df["timestamp"].max()


In [ ]:
ticks_df["mid"] = (ticks_df["bid"] + ticks_df["ask"]) / 2
ticks_df["volume"] = ticks_df["bid_volume"] + ticks_df["ask_volume"]

ohlcv_1m = (
    ticks_df
    .set_index("timestamp")
    .resample("1min")
    .agg(
        open=("mid", "first"),
        high=("mid", "max"),
        low=("mid", "min"),
        close=("mid", "last"),
        volume=("volume", "sum"),
    )
    .dropna()
    .reset_index()
)

ohlcv_1m.head()


In [ ]:
ohlcv_1m.shape, ohlcv_1m["timestamp"].min(), ohlcv_1m["timestamp"].max()


In [ ]:
import lzma
import struct
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd


def decode_dukascopy_bi5(
    bi5_path: Path,
    hour_start: datetime,
    price_scale: int,
) -> pd.DataFrame:
    compressed = bi5_path.read_bytes()
    decompressed = lzma.decompress(compressed)

    record_size = 20

    if len(decompressed) % record_size != 0:
        raise ValueError("Invalid Dukascopy BI5 file size.")

    rows = []

    for offset in range(0, len(decompressed), record_size):
        time_delta_ms, ask, bid, ask_volume, bid_volume = struct.unpack(
            ">iii ff",
            decompressed[offset : offset + record_size],
        )

        rows.append(
            {
                "timestamp": hour_start + timedelta(milliseconds=time_delta_ms),
                "bid": bid / price_scale,
                "ask": ask / price_scale,
                "bid_volume": bid_volume,
                "ask_volume": ask_volume,
            }
        )

    return pd.DataFrame(rows)


In [ ]:
decoded_df = decode_dukascopy_bi5(
    bi5_path=bi5_path,
    hour_start=ts,
    price_scale=100000,
)

decoded_df.head()


In [ ]:
decoded_df.shape, decoded_df["timestamp"].min(), decoded_df["timestamp"].max()


In [ ]:
from transformers.dukascopy_ticks import decode_dukascopy_bi5


decoded_df = decode_dukascopy_bi5(
    bi5_path=bi5_path,
    hour_start=ts,
    price_scale=100000,
)

decoded_df.shape, decoded_df["timestamp"].min(), decoded_df["timestamp"].max()


In [ ]:
from datetime import datetime, timezone
from pathlib import Path

from providers.dukascopy import DukascopyProvider, DukascopyRequest


provider = DukascopyProvider()

request = DukascopyRequest(
    symbol="EURUSD",
    hour_start=datetime(2024, 1, 2, 12, tzinfo=timezone.utc),
)

result = provider.download(
    request=request,
    output_dir=Path("data/staging/dukascopy/eurusd/2024/01"),
)

result


In [ ]:
result.file.exists(), result.file.stat().st_size, result.is_empty


In [ ]:
from datetime import datetime, timezone, timedelta
from pathlib import Path

from providers.dukascopy import DukascopyProvider, DukascopyRequest


provider = DukascopyProvider()

symbol = "EURUSD"
asset = "eurusd"
day_start = datetime(2024, 1, 2, 0, tzinfo=timezone.utc)

output_dir = Path("data/staging/dukascopy") / asset / "2024" / "01"

download_results = []
errors = []

for hour in range(24):
    hour_start = day_start + timedelta(hours=hour)

    request = DukascopyRequest(
        symbol=symbol,
        hour_start=hour_start,
    )

    try:
        result = provider.download(
            request=request,
            output_dir=output_dir,
        )

        download_results.append(result)

    except FileNotFoundError as error:
        errors.append(
            {
                "hour_start": hour_start,
                "error": str(error),
                "type": "not_found",
            }
        )

    except RuntimeError as error:
        errors.append(
            {
                "hour_start": hour_start,
                "error": str(error),
                "type": "runtime",
            }
        )

len(download_results), len(errors), errors[:5]


In [ ]:
non_empty_results = [
    result
    for result in download_results
    if not result.is_empty
]

empty_results = [
    result
    for result in download_results
    if result.is_empty
]

len(non_empty_results), len(empty_results)


In [ ]:
[(result.file.name, result.file.stat().st_size) for result in download_results[:5]]


In [ ]:
import pandas as pd

from transformers.dukascopy_ticks import decode_dukascopy_bi5


daily_tick_frames = []

for result in download_results:
    if result.is_empty:
        continue

    hour_start_text = result.file.name.replace("EURUSD-", "").replace("_ticks.bi5", "")
    hour_start = datetime.strptime(hour_start_text, "%Y-%m-%d-%H").replace(
        tzinfo=timezone.utc
    )

    tick_df = decode_dukascopy_bi5(
        bi5_path=result.file,
        hour_start=hour_start,
        price_scale=100000,
    )

    daily_tick_frames.append(tick_df)

daily_ticks_df = pd.concat(
    daily_tick_frames,
    ignore_index=True,
).sort_values("timestamp").reset_index(drop=True)

daily_ticks_df.shape, daily_ticks_df["timestamp"].min(), daily_ticks_df["timestamp"].max()


In [ ]:
import importlib
import providers.dukascopy

importlib.reload(providers.dukascopy)

from providers.dukascopy import DukascopyProvider, DukascopyRequest


In [ ]:
from datetime import datetime, timezone
from pathlib import Path


provider = DukascopyProvider()

request = DukascopyRequest(
    symbol="EURUSD",
    hour_start=datetime(2024, 1, 2, 12, tzinfo=timezone.utc),
)

test_result = provider.download(
    request=request,
    output_dir=Path("data/staging/dukascopy/eurusd/2024/01"),
)

test_result


In [ ]:
test_result.hour_start, test_result.file.exists(), test_result.file.stat().st_size, test_result.is_empty


In [ ]:
from datetime import datetime, timezone, timedelta
from pathlib import Path

import pandas as pd

from providers.dukascopy import DukascopyProvider, DukascopyRequest
from transformers.dukascopy_ticks import decode_dukascopy_bi5


provider = DukascopyProvider()

symbol = "EURUSD"
asset = "eurusd"
day_start = datetime(2024, 1, 2, 0, tzinfo=timezone.utc)

output_dir = Path("data/staging/dukascopy") / asset / "2024" / "01"

download_results = []
errors = []

for hour in range(24):
    hour_start = day_start + timedelta(hours=hour)

    request = DukascopyRequest(
        symbol=symbol,
        hour_start=hour_start,
    )

    try:
        result = provider.download(
            request=request,
            output_dir=output_dir,
        )
        download_results.append(result)

    except Exception as error:
        errors.append(
            {
                "hour_start": hour_start,
                "error": str(error),
            }
        )

tick_frames = []

for result in download_results:
    if result.is_empty:
        continue

    tick_df = decode_dukascopy_bi5(
        bi5_path=result.file,
        hour_start=result.hour_start,
        price_scale=100000,
    )

    tick_frames.append(tick_df)

daily_ticks_df = (
    pd.concat(tick_frames, ignore_index=True)
    .sort_values("timestamp")
    .reset_index(drop=True)
)

daily_ticks_df.shape, daily_ticks_df["timestamp"].min(), daily_ticks_df["timestamp"].max(), len(errors)


In [ ]:
from datetime import datetime, timezone, timedelta
from pathlib import Path

import pandas as pd

from providers.dukascopy import DukascopyProvider, DukascopyRequest
from transformers.dukascopy_ticks import decode_dukascopy_bi5


provider = DukascopyProvider()

symbol = "EURUSD"
asset = "eurusd"
price_scale = 100000

day_start = datetime(2024, 1, 2, 0, tzinfo=timezone.utc)
output_dir = Path("data/staging/dukascopy") / asset / "2024" / "01"

download_results = []
errors = []

for hour in range(24):
    hour_start = day_start + timedelta(hours=hour)

    request = DukascopyRequest(
        symbol=symbol,
        hour_start=hour_start,
    )

    try:
        result = provider.download(
            request=request,
            output_dir=output_dir,
        )
        download_results.append(result)

    except Exception as error:
        errors.append(
            {
                "hour_start": hour_start,
                "error": str(error),
            }
        )

len(download_results), len(errors), errors[:3]


In [ ]:
[(r.hour_start, r.file.stat().st_size, r.is_empty) for r in download_results[:5]]


In [ ]:
tick_frames = []
decode_errors = []

for result in download_results:
    if result.is_empty:
        continue

    try:
        tick_df = decode_dukascopy_bi5(
            bi5_path=result.file,
            hour_start=result.hour_start,
            price_scale=price_scale,
        )

        if not tick_df.empty:
            tick_frames.append(tick_df)

    except Exception as error:
        decode_errors.append(
            {
                "hour_start": result.hour_start,
                "file": str(result.file),
                "error": str(error),
            }
        )

len(tick_frames), len(decode_errors), decode_errors[:3]


In [ ]:
daily_ticks_df = (
    pd.concat(tick_frames, ignore_index=True)
    .sort_values("timestamp")
    .reset_index(drop=True)
)

daily_ticks_df.shape, daily_ticks_df["timestamp"].min(), daily_ticks_df["timestamp"].max()


In [ ]:
import importlib
import transformers.dukascopy_ticks

importlib.reload(transformers.dukascopy_ticks)

from transformers.dukascopy_ticks import decode_dukascopy_downloads


daily_ticks_df = decode_dukascopy_downloads(
    downloads=download_results,
    price_scale=100000,
)

daily_ticks_df.shape, daily_ticks_df["timestamp"].min(), daily_ticks_df["timestamp"].max()


In [ ]:
daily_ticks_df.head()



In [ ]:
from pathlib import Path


raw_tick_parquet_path = Path(
    "data/raw/dukascopy/eurusd/2024/01/EURUSD-ticks-2024-01-02.parquet"
)

raw_tick_parquet_path.parent.mkdir(parents=True, exist_ok=True)

daily_ticks_df.to_parquet(
    raw_tick_parquet_path,
    index=False,
    engine="pyarrow",
)

raw_tick_parquet_path


In [ ]:
raw_tick_parquet_path.exists(), raw_tick_parquet_path.stat().st_size


In [ ]:
from uploaders.azure_blob import upload_file


raw_tick_blob_name = "raw/dukascopy/eurusd/2024/01/EURUSD-ticks-2024-01-02.parquet"

upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=raw_tick_parquet_path,
    blob_name=raw_tick_blob_name,
)


In [ ]:
from pathlib import Path

from uploaders.azure_blob import init_azure_client


blob_service_client, container_name = init_azure_client(
    env_path=Path(".env"),
)

container_name


In [ ]:
from uploaders.azure_blob import upload_file


raw_tick_blob_name = "raw/dukascopy/eurusd/2024/01/EURUSD-ticks-2024-01-02.parquet"

upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=raw_tick_parquet_path,
    blob_name=raw_tick_blob_name,
)


In [ ]:
from uploaders.azure_blob import blob_exists


blob_exists(
    blob_service_client=blob_service_client,
    container_name=container_name,
    blob_name=raw_tick_blob_name,
)


In [ ]:
from writers.dukascopy_raw_parquet import write_dukascopy_raw_ticks_to_parquet


In [ ]:
raw_tick_parquet_path = write_dukascopy_raw_ticks_to_parquet(
    df=daily_ticks_df,
    output_path=Path("data/raw/dukascopy/eurusd/2024/01/EURUSD-ticks-2024-01-02.parquet"),
)

raw_tick_parquet_path


In [ ]:
import importlib
import pipelines.dukascopy_pipeline

importlib.reload(pipelines.dukascopy_pipeline)

from pipelines.dukascopy_pipeline import download_dukascopy_day


In [ ]:
from datetime import datetime, timezone
from pathlib import Path


downloads, errors = download_dukascopy_day(
    symbol="EURUSD",
    day_start=datetime(2024, 1, 2, 0, tzinfo=timezone.utc),
    output_dir=Path("data/staging/dukascopy/eurusd/2024/01"),
)

len(downloads), len(errors)


In [ ]:
import importlib
import pipelines.dukascopy_pipeline

importlib.reload(pipelines.dukascopy_pipeline)

from pipelines.dukascopy_pipeline import (
    download_dukascopy_day,
    build_dukascopy_day_raw_ticks,
)


In [ ]:
daily_ticks_df = build_dukascopy_day_raw_ticks(
    downloads=downloads,
    price_scale=100000,
)

daily_ticks_df.shape, daily_ticks_df["timestamp"].min(), daily_ticks_df["timestamp"].max()


In [ ]:
import importlib
import pipelines.dukascopy_pipeline

importlib.reload(pipelines.dukascopy_pipeline)

from pipelines.dukascopy_pipeline import (
    download_dukascopy_day,
    build_dukascopy_day_raw_ticks,
    write_and_upload_dukascopy_raw_ticks,
)


In [ ]:
from pathlib import Path


local_raw_path = Path("data/raw/dukascopy/eurusd/2024/01/EURUSD-ticks-2024-01-02.parquet")
raw_blob_name = "raw/dukascopy/eurusd/2024/01/EURUSD-ticks-2024-01-02.parquet"

raw_path = write_and_upload_dukascopy_raw_ticks(
    ticks_df=daily_ticks_df,
    local_raw_path=local_raw_path,
    raw_blob_name=raw_blob_name,
    blob_service_client=blob_service_client,
    container_name=container_name,
)

raw_path


In [ ]:
from uploaders.azure_blob import blob_exists


blob_exists(
    blob_service_client=blob_service_client,
    container_name=container_name,
    blob_name=raw_blob_name,
)


In [ ]:
import importlib
import pipelines.dukascopy_pipeline

importlib.reload(pipelines.dukascopy_pipeline)


In [ ]:
from pipelines.dukascopy_pipeline import (
    download_dukascopy_day,
    build_dukascopy_day_raw_ticks,
)


In [ ]:
from datetime import datetime, timezone
from pathlib import Path

from pipelines.dukascopy_pipeline import download_dukascopy_day


downloads, errors = download_dukascopy_day(
    symbol="EURUSD",
    day_start=datetime(2024, 1, 2, 0, tzinfo=timezone.utc),
    output_dir=Path("data/staging/dukascopy/eurusd/2024/01"),
    timeout_sec=10,
    max_attempts=2,
    retry_sleep_sec=2,
)

len(downloads), len(errors), errors[:3]


In [ ]:
from pipelines.dukascopy_pipeline import build_dukascopy_day_raw_ticks


daily_ticks_df = build_dukascopy_day_raw_ticks(
    downloads=downloads,
    price_scale=100000,
)

daily_ticks_df.shape, daily_ticks_df["timestamp"].min(), daily_ticks_df["timestamp"].max()


In [ ]:
from pathlib import Path

from pipelines.dukascopy_pipeline import write_and_upload_dukascopy_raw_ticks
from uploaders.azure_blob import init_azure_client, blob_exists


blob_service_client, container_name = init_azure_client(
    env_path=Path(".env"),
)

local_raw_path = Path(
    "data/raw/dukascopy/eurusd/2024/01/EURUSD-ticks-2024-01-02.parquet"
)

raw_blob_name = "raw/dukascopy/eurusd/2024/01/EURUSD-ticks-2024-01-02.parquet"

raw_path = write_and_upload_dukascopy_raw_ticks(
    ticks_df=daily_ticks_df,
    local_raw_path=local_raw_path,
    raw_blob_name=raw_blob_name,
    blob_service_client=blob_service_client,
    container_name=container_name,
)

blob_exists(
    blob_service_client=blob_service_client,
    container_name=container_name,
    blob_name=raw_blob_name,
)


In [ ]:
import importlib
import pipelines.dukascopy_pipeline

importlib.reload(pipelines.dukascopy_pipeline)

from pipelines.dukascopy_pipeline import build_dukascopy_raw_ticks_for_window


In [ ]:
import importlib
import pipelines.dukascopy_pipeline

importlib.reload(pipelines.dukascopy_pipeline)

from pipelines.dukascopy_pipeline import build_dukascopy_raw_ticks_for_window


In [ ]:
from datetime import date
from pathlib import Path

from planner.time_windows import TimeWindow
from pipelines.dukascopy_pipeline import build_dukascopy_raw_ticks_for_window


test_window = TimeWindow(
    start_date=date(2024, 1, 2),
    end_date=date(2024, 1, 3),
)

ticks_df, errors = build_dukascopy_raw_ticks_for_window(
    symbol="EURUSD",
    window=test_window,
    staging_dir=Path("data/staging/dukascopy/eurusd/2024/01"),
    price_scale=100000,
    timeout_sec=5,
    max_attempts=2,
    retry_sleep_sec=1,
    print_progress=True,
)

ticks_df.shape, ticks_df["timestamp"].min(), ticks_df["timestamp"].max(), len(errors)


In [ ]:
from uploaders.azure_blob import init_azure_client, blob_exists
from pipelines.dukascopy_pipeline import write_and_upload_dukascopy_raw_ticks


blob_service_client, container_name = init_azure_client(
    env_path=Path(".env"),
)

local_raw_path = Path(
    "data/raw/dukascopy/eurusd/2024/01/EURUSD-ticks-2024-01-02_to_2024-01-03.parquet"
)

raw_blob_name = "raw/dukascopy/eurusd/2024/01/EURUSD-ticks-2024-01-02_to_2024-01-03.parquet"

raw_path = write_and_upload_dukascopy_raw_ticks(
    ticks_df=ticks_df,
    local_raw_path=local_raw_path,
    raw_blob_name=raw_blob_name,
    blob_service_client=blob_service_client,
    container_name=container_name,
)

blob_exists(
    blob_service_client=blob_service_client,
    container_name=container_name,
    blob_name=raw_blob_name,
)


In [ ]:
import importlib
import paths.data_paths

importlib.reload(paths.data_paths)

from paths.data_paths import build_dukascopy_raw_tick_paths


dukascopy_paths = build_dukascopy_raw_tick_paths(
    broker="dukascopy",
    asset="EURUSD",
    broker_symbol="EURUSD",
    year=2024,
    month=1,
)

dukascopy_paths


In [ ]:
from paths.data_paths import build_dukascopy_raw_tick_paths
from pipelines.dukascopy_pipeline import write_and_upload_dukascopy_raw_ticks


dukascopy_paths = build_dukascopy_raw_tick_paths(
    broker="dukascopy",
    asset="EURUSD",
    broker_symbol="EURUSD",
    year=2024,
    month=1,
)

raw_path = write_and_upload_dukascopy_raw_ticks(
    ticks_df=ticks_df,
    local_raw_path=dukascopy_paths.local_raw_tick_file,
    raw_blob_name=dukascopy_paths.raw_tick_blob_name,
    blob_service_client=blob_service_client,
    container_name=container_name,
)

raw_path


In [ ]:
blob_exists(
    blob_service_client=blob_service_client,
    container_name=container_name,
    blob_name=dukascopy_paths.raw_tick_blob_name,
)


In [ ]:
import importlib
import transformers.dukascopy_ohlcv

importlib.reload(transformers.dukascopy_ohlcv)

from transformers.dukascopy_ohlcv import transform_dukascopy_ticks_to_ohlcv


In [ ]:
dukascopy_ohlcv_df = transform_dukascopy_ticks_to_ohlcv(
    ticks_df=ticks_df,
    interval="1min",
)

dukascopy_ohlcv_df.head()


In [ ]:
dukascopy_ohlcv_df.shape, dukascopy_ohlcv_df["timestamp"].min(), dukascopy_ohlcv_df["timestamp"].max()


In [ ]:
import importlib
import paths.data_paths

importlib.reload(paths.data_paths)

from paths.data_paths import build_dukascopy_bronze_paths


bronze_paths = build_dukascopy_bronze_paths(
    broker="dukascopy",
    asset="EURUSD",
    broker_symbol="EURUSD",
    interval="1m",
    year=2024,
    month=1,
)

bronze_paths


In [ ]:
from writers.ohlcv_parquet import write_ohlcv_to_parquet
from uploaders.azure_blob import upload_file, blob_exists


bronze_path = write_ohlcv_to_parquet(
    df=dukascopy_ohlcv_df,
    output_path=bronze_paths.local_bronze_parquet_file,
)

upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=bronze_path,
    blob_name=bronze_paths.bronze_parquet_blob_name,
)

blob_exists(
    blob_service_client=blob_service_client,
    container_name=container_name,
    blob_name=bronze_paths.bronze_parquet_blob_name,
)


In [ ]:
from metadata.manifest import build_manifest, write_manifest
from uploaders.azure_blob import upload_file, blob_exists


manifest = build_manifest(
    provider="dukascopy",
    symbol="EURUSD",
    interval="1m",
    year=2024,
    month=1,
    source_url="multiple Dukascopy hourly .bi5 files",
    raw_file_path=dukascopy_paths.local_raw_tick_file,
    parquet_file_path=bronze_paths.local_bronze_parquet_file,
    validation_result=type(
        "ValidationResult",
        (),
        {
            "row_count": len(ticks_df),
            "bad_column_count": 0,
            "bad_timestamp_count": 0,
            "bad_number_count": 0,
            "duplicate_timestamp_count": int(ticks_df["timestamp"].duplicated().sum()),
            "is_time_ordered": bool(ticks_df["timestamp"].is_monotonic_increasing),
            "min_timestamp": ticks_df["timestamp"].min(),
            "max_timestamp": ticks_df["timestamp"].max(),
        },
    )(),
    ohlcv_df=dukascopy_ohlcv_df,
)

manifest_path = write_manifest(
    manifest=manifest,
    output_path=bronze_paths.local_manifest_file,
)

upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=manifest_path,
    blob_name=bronze_paths.manifest_blob_name,
)

blob_exists(
    blob_service_client=blob_service_client,
    container_name=container_name,
    blob_name=bronze_paths.manifest_blob_name,
)


In [ ]:
import importlib
import validators.dukascopy_ticks

importlib.reload(validators.dukascopy_ticks)

from validators.dukascopy_ticks import validate_dukascopy_ticks


dukascopy_validation_result = validate_dukascopy_ticks(ticks_df)

dukascopy_validation_result


In [ ]:
from metadata.manifest import build_manifest, write_manifest
from uploaders.azure_blob import upload_file, blob_exists


manifest = build_manifest(
    provider="dukascopy",
    symbol="EURUSD",
    interval="1m",
    year=2024,
    month=1,
    source_url="multiple Dukascopy hourly .bi5 files",
    raw_file_path=dukascopy_paths.local_raw_tick_file,
    parquet_file_path=bronze_paths.local_bronze_parquet_file,
    validation_result=dukascopy_validation_result,
    ohlcv_df=dukascopy_ohlcv_df,
)

manifest_path = write_manifest(
    manifest=manifest,
    output_path=bronze_paths.local_manifest_file,
)

upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=manifest_path,
    blob_name=bronze_paths.manifest_blob_name,
)

blob_exists(
    blob_service_client=blob_service_client,
    container_name=container_name,
    blob_name=bronze_paths.manifest_blob_name,
)


In [ ]:
from metadata.success_marker import write_success_marker
from uploaders.azure_blob import upload_file, blob_exists


success_path = write_success_marker(
    output_path=bronze_paths.local_success_file,
)

upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=success_path,
    blob_name=bronze_paths.success_blob_name,
)

blob_exists(
    blob_service_client=blob_service_client,
    container_name=container_name,
    blob_name=bronze_paths.success_blob_name,
)


In [ ]:
from pathlib import Path

from config_loader.csv_config import (
    load_broker_strategy,
    load_broker_asset_matrix,
    load_download_period,
)
from planner.download_plan import build_download_plan


config_dir = Path("config")

broker_strategy_df = load_broker_strategy(config_dir)
broker_asset_matrix_df = load_broker_asset_matrix(config_dir)
download_period_df = load_download_period(config_dir)

plan = build_download_plan(
    broker_strategy_df=broker_strategy_df,
    broker_asset_matrix_df=broker_asset_matrix_df,
    download_period_df=download_period_df,
)

len(plan)


In [ ]:
dukascopy_plan = [
    item
    for item in plan
    if item.broker == "dukascopy"
]

len(dukascopy_plan), dukascopy_plan[:5]


In [ ]:
broker_strategy_df


In [ ]:
eurusd_item = next(
    item
    for item in dukascopy_plan
    if item.asset == "EURUSD"
)

eurusd_item


In [ ]:
from pathlib import Path

from pipelines.dukascopy_pipeline import build_dukascopy_raw_ticks_for_window


ticks_df, errors = build_dukascopy_raw_ticks_for_window(
    symbol=eurusd_item.broker_symbol,
    window=eurusd_item.window,
    staging_dir=Path("data/staging/dukascopy/eurusd/2024/01"),
    price_scale=100000,
    timeout_sec=5,
    max_attempts=2,
    retry_sleep_sec=1,
    print_progress=True,
)


In [ ]:
ticks_df.shape, ticks_df["timestamp"].min(), ticks_df["timestamp"].max(), len(errors)


In [ ]:
from paths.data_paths import build_dukascopy_raw_tick_paths
from pipelines.dukascopy_pipeline import write_and_upload_dukascopy_raw_ticks
from uploaders.azure_blob import blob_exists


dukascopy_raw_paths = build_dukascopy_raw_tick_paths(
    broker=eurusd_item.broker,
    asset=eurusd_item.asset,
    broker_symbol=eurusd_item.broker_symbol,
    year=eurusd_item.window.start_date.year,
    month=eurusd_item.window.start_date.month,
)

raw_path = write_and_upload_dukascopy_raw_ticks(
    ticks_df=ticks_df,
    local_raw_path=dukascopy_raw_paths.local_raw_tick_file,
    raw_blob_name=dukascopy_raw_paths.raw_tick_blob_name,
    blob_service_client=blob_service_client,
    container_name=container_name,
)

blob_exists(
    blob_service_client=blob_service_client,
    container_name=container_name,
    blob_name=dukascopy_raw_paths.raw_tick_blob_name,
)


In [ ]:
from transformers.dukascopy_ohlcv import transform_dukascopy_ticks_to_ohlcv
from writers.ohlcv_parquet import write_ohlcv_to_parquet
from paths.data_paths import build_dukascopy_bronze_paths
from uploaders.azure_blob import upload_file, blob_exists


dukascopy_ohlcv_df = transform_dukascopy_ticks_to_ohlcv(
    ticks_df=ticks_df,
    interval="1min",
)

dukascopy_bronze_paths = build_dukascopy_bronze_paths(
    broker=eurusd_item.broker,
    asset=eurusd_item.asset,
    broker_symbol=eurusd_item.broker_symbol,
    interval="1m",
    year=eurusd_item.window.start_date.year,
    month=eurusd_item.window.start_date.month,
)

bronze_path = write_ohlcv_to_parquet(
    df=dukascopy_ohlcv_df,
    output_path=dukascopy_bronze_paths.local_bronze_parquet_file,
)

upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=bronze_path,
    blob_name=dukascopy_bronze_paths.bronze_parquet_blob_name,
)

blob_exists(
    blob_service_client=blob_service_client,
    container_name=container_name,
    blob_name=dukascopy_bronze_paths.bronze_parquet_blob_name,
)


In [ ]:
from validators.dukascopy_ticks import validate_dukascopy_ticks
from metadata.manifest import build_manifest, write_manifest
from uploaders.azure_blob import upload_file, blob_exists


dukascopy_validation_result = validate_dukascopy_ticks(ticks_df)

manifest = build_manifest(
    provider=eurusd_item.broker,
    symbol=eurusd_item.broker_symbol,
    interval="1m",
    year=eurusd_item.window.start_date.year,
    month=eurusd_item.window.start_date.month,
    source_url="multiple Dukascopy hourly .bi5 files",
    raw_file_path=dukascopy_raw_paths.local_raw_tick_file,
    parquet_file_path=dukascopy_bronze_paths.local_bronze_parquet_file,
    validation_result=dukascopy_validation_result,
    ohlcv_df=dukascopy_ohlcv_df,
)

manifest_path = write_manifest(
    manifest=manifest,
    output_path=dukascopy_bronze_paths.local_manifest_file,
)

upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=manifest_path,
    blob_name=dukascopy_bronze_paths.manifest_blob_name,
)

blob_exists(
    blob_service_client=blob_service_client,
    container_name=container_name,
    blob_name=dukascopy_bronze_paths.manifest_blob_name,
)


In [ ]:
from metadata.success_marker import write_success_marker


success_path = write_success_marker(
    output_path=dukascopy_bronze_paths.local_success_file,
)

upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=success_path,
    blob_name=dukascopy_bronze_paths.success_blob_name,
)

blob_exists(
    blob_service_client=blob_service_client,
    container_name=container_name,
    blob_name=dukascopy_bronze_paths.success_blob_name,
)


In [ ]:
import importlib
import pipelines.dukascopy_pipeline

importlib.reload(pipelines.dukascopy_pipeline)

from pipelines.dukascopy_pipeline import run_dukascopy_plan_item


In [ ]:
result = run_dukascopy_plan_item(
    item=eurusd_item,
    interval="1m",
    price_scale=100000,
    blob_service_client=blob_service_client,
    container_name=container_name,
    timeout_sec=5,
    max_attempts=2,
    retry_sleep_sec=1,
    print_progress=True,
)

result


In [ ]:
eurusd_items = [
    item
    for item in dukascopy_plan
    if item.asset == "EURUSD"
]

eurusd_items[:3]


In [ ]:
result = run_dukascopy_plan_item(
    item=eurusd_items[1],
    interval="1m",
    price_scale=100000,
    blob_service_client=blob_service_client,
    container_name=container_name,
    timeout_sec=5,
    max_attempts=2,
    retry_sleep_sec=1,
    print_progress=True,
)

result


In [ ]:
import importlib
import pipelines.dukascopy_pipeline

importlib.reload(pipelines.dukascopy_pipeline)

from pipelines.dukascopy_pipeline import run_dukascopy_plan_item


In [ ]:
result = run_dukascopy_plan_item(
    item=eurusd_items[1],
    interval="1m",
    price_scale=100000,
    blob_service_client=blob_service_client,
    container_name=container_name,
    timeout_sec=5,
    max_attempts=2,
    retry_sleep_sec=1,
    print_progress=True,
)

result


In [ ]:
result


In [ ]:
import importlib
import pipelines.dukascopy_pipeline

importlib.reload(pipelines.dukascopy_pipeline)

from pipelines.dukascopy_pipeline import run_dukascopy_plan


In [ ]:
eurusd_items = [
    item
    for item in dukascopy_plan
    if item.asset == "EURUSD"
]

results = run_dukascopy_plan(
    plan=eurusd_items[:2],
    interval="1m",
    price_scale_by_asset={
        "EURUSD": 100000,
    },
    blob_service_client=blob_service_client,
    container_name=container_name,
    timeout_sec=5,
    max_attempts=2,
    retry_sleep_sec=1,
    print_progress=True,
)

results


In [ ]:
import importlib
import config_loader.csv_config

importlib.reload(config_loader.csv_config)


In [ ]:
from pathlib import Path

from config_loader.csv_config import load_broker_asset_settings


broker_asset_settings_df = load_broker_asset_settings(Path("config"))

broker_asset_settings_df


In [ ]:
dukascopy_price_scale_by_asset = {
    row["asset"]: int(row["price_scale"])
    for _, row in broker_asset_settings_df[
        broker_asset_settings_df["broker"] == "dukascopy"
    ].iterrows()
}

dukascopy_price_scale_by_asset


In [ ]:
import importlib
import pipelines.dukascopy_runner

importlib.reload(pipelines.dukascopy_runner)

from pipelines.dukascopy_runner import run_dukascopy_from_config


In [ ]:
results = run_dukascopy_from_config(
    start_index=2,
    limit=1,
    timeout_sec=5,
    max_attempts=2,
    retry_sleep_sec=1,
    print_progress=True,
)

results


In [ ]:
results = run_dukascopy_from_config(
    start_index=8,
    limit=1,
    timeout_sec=5,
    max_attempts=2,
    retry_sleep_sec=1,
    print_progress=True,
)

results


In [ ]:
results = run_dukascopy_from_config(
    start_index=14,
    limit=1,
    timeout_sec=5,
    max_attempts=2,
    retry_sleep_sec=1,
    print_progress=True,
)

results


In [ ]:
from pipelines.dukascopy_runner import run_dukascopy_from_config


# 2024-01 Dukascopy itemek:
# 0 = XAUUSD
# 1 = XAGUSD
# 2 = EURUSD
# 3 = US500
# 4 = DAX
# 5 = BTCUSD

results = run_dukascopy_from_config(
    start_index=5,
    limit=1,
    interval="1m",
    timeout_sec=5,
    max_attempts=2,
    retry_sleep_sec=1,
    print_progress=True,
)

results


# Végleges letöltők

### Dukascopy

In [ ]:
from pipelines.dukascopy_runner import run_dukascopy_from_config

results = run_dukascopy_from_config(
    interval="1m",
    print_progress=True,
    timeout_sec=10,
    max_attempts=2,
    retry_sleep_sec=2,
)

results

### Binance

In [ ]:
from pipelines.binance_runner import run_binance_from_config

test_results = run_binance_from_config(
    interval="1m",
    start_index=0,
    limit=1,
    max_attempts=2,
    retry_sleep_sec=2,
)

test_results


In [ ]:
from pipelines.binance_runner import run_binance_from_config

test_results = run_binance_from_config(
    interval="1m",
    start_index=0,
    limit=1,
    max_attempts=2,
    retry_sleep_sec=2,
)

test_results


In [ ]:
print("ok")


In [ ]:
from pathlib import Path

from pipelines.binance_pipeline import download_binance_with_retry
from providers.binance import BinanceRequest


class AlwaysFailProvider:
    def __init__(self):
        self.calls = 0

    def download(self, request, output_dir):
        self.calls += 1
        raise RuntimeError("fake error")


provider = AlwaysFailProvider()

request = BinanceRequest(
    symbol="BTCUSDT",
    interval="1m",
    year=2024,
    month=1,
)

result, error = download_binance_with_retry(
    provider=provider,
    request=request,
    output_dir=Path("data/test"),
    max_attempts=2,
    retry_sleep_sec=0,
)

print(result)
print(error)
print(provider.calls)


In [ ]:
print("kernel ok")



In [ ]:
import sys
print(sys.executable)


In [ ]:
%pip install -r requirements.txt

In [ ]:
import azure.storage.blob
import pandas
import pyarrow
import pipelines.binance_pipeline
import pipelines.dukascopy_pipeline

print("onlab kernel ok")


In [ ]:
import sys
print(sys.executable)


In [ ]:
import importlib
import pipelines.binance_pipeline

importlib.reload(pipelines.binance_pipeline)

from pathlib import Path
from pipelines.binance_pipeline import download_binance_with_retry
from providers.binance import BinanceRequest


class AlwaysFailProvider:
    def __init__(self):
        self.calls = 0

    def download(self, request, output_dir):
        self.calls += 1
        raise RuntimeError("fake error")


provider = AlwaysFailProvider()

request = BinanceRequest(
    symbol="BTCUSDT",
    interval="1m",
    year=2024,
    month=1,
)

result, error = download_binance_with_retry(
    provider=provider,
    request=request,
    output_dir=Path("data/test"),
    max_attempts=2,
    retry_sleep_sec=0,
)

print(result)
print(error)
print(provider.calls)


In [ ]:
import pandas as pd

from transformers.dukascopy_ohlcv import transform_dukascopy_ticks_to_ohlcv


ticks_df = pd.DataFrame(
    {
        "timestamp": pd.to_datetime(
            [
                "2024-01-01 00:00:00",
                "2024-01-01 00:00:30",
                "2024-01-01 00:01:00",
                "2024-01-01 00:01:30",
                "2024-01-01 00:02:00",
            ],
            utc=True,
        ),
        "bid": [100, 101, 102, 103, 104],
        "ask": [102, 103, 104, 105, 106],
        "bid_volume": [1, 1, 1, 1, 1],
        "ask_volume": [2, 2, 2, 2, 2],
    }
)

ohlcv_1min = transform_dukascopy_ticks_to_ohlcv(
    ticks_df=ticks_df,
    interval="1min",
)

ohlcv_2min = transform_dukascopy_ticks_to_ohlcv(
    ticks_df=ticks_df,
    interval="2min",
)

print(len(ohlcv_1min))
print(len(ohlcv_2min))
print(ohlcv_1min[["timestamp", "open", "close", "volume"]])
print(ohlcv_2min[["timestamp", "open", "close", "volume"]])


In [ ]:
import pandas as pd

from pathlib import Path
from datetime import datetime, timezone

from metadata.manifest import build_manifest


class FakeValidationResult:
    row_count = 0
    bad_column_count = 0
    bad_timestamp_count = 0
    bad_number_count = 0
    duplicate_timestamp_count = 0
    is_time_ordered = True
    min_timestamp = None
    max_timestamp = None


empty_ohlcv_df = pd.DataFrame(
    columns=["timestamp", "open", "high", "low", "close", "volume"]
)

manifest = build_manifest(
    provider="test",
    symbol="TEST",
    interval="1m",
    year=2024,
    month=1,
    source_url="test-url",
    raw_file_path=Path("raw.csv"),
    parquet_file_path=Path("bronze.parquet"),
    validation_result=FakeValidationResult(),
    ohlcv_df=empty_ohlcv_df,
    ingestion_errors=[
        {
            "hour_start": datetime(2024, 1, 1, tzinfo=timezone.utc),
            "error": "fake error",
        }
    ],
)

manifest["data"], manifest["ingestion"]


In [ ]:
import importlib
import metadata.manifest

importlib.reload(metadata.manifest)

from metadata.manifest import build_manifest


In [ ]:
import importlib

import metadata.manifest
import pipelines.binance_pipeline
import pipelines.binance_runner
import pipelines.dukascopy_pipeline
import pipelines.dukascopy_runner

importlib.reload(metadata.manifest)
importlib.reload(pipelines.binance_pipeline)
importlib.reload(pipelines.binance_runner)
importlib.reload(pipelines.dukascopy_pipeline)
importlib.reload(pipelines.dukascopy_runner)

print("pipeline modules ok")


In [ ]:
import importlib

import planner.download_plan
import pipelines.binance_runner

importlib.reload(planner.download_plan)
importlib.reload(pipelines.binance_runner)

print("binance month range code loaded")


In [ ]:
from pipelines.binance_runner import _build_month_date_range

print(_build_month_date_range(None, None))
print(_build_month_date_range("2024-01", "2024-03"))


In [ ]:
_build_month_date_range("2026-05", "2026-05")


In [ ]:
from pathlib import Path

from config_loader.csv_config import (
    load_broker_asset_matrix,
    load_broker_strategy,
    load_download_period,
)
from planner.download_plan import build_download_plan
from pipelines.binance_runner import _build_month_date_range


config_dir = Path("config")

broker_strategy_df = load_broker_strategy(config_dir)
broker_asset_matrix_df = load_broker_asset_matrix(config_dir)
download_period_df = load_download_period(config_dir)

start_date, end_date = _build_month_date_range("2024-01", "2024-03")

plan = build_download_plan(
    broker_strategy_df=broker_strategy_df,
    broker_asset_matrix_df=broker_asset_matrix_df,
    download_period_df=download_period_df,
    start_date=start_date,
    end_date=end_date,
)

binance_plan = [
    item
    for item in plan
    if item.broker == "binance"
]

[
    (item.broker, item.asset, item.broker_symbol, item.window.start_date, item.window.end_date)
    for item in binance_plan
]


In [ ]:
from pipelines.binance_runner import run_binance_from_config

results = run_binance_from_config(
    start_month="2024-01",
    end_month="2024-03",
    interval="1m",
)


In [ ]:
import pandas as pd

pd.DataFrame([result.__dict__ for result in results])


In [ ]:
import importlib

import pipelines.dukascopy_runner

importlib.reload(pipelines.dukascopy_runner)

print("dukascopy month range code loaded")


In [ ]:
from pathlib import Path

from config_loader.csv_config import (
    load_broker_asset_matrix,
    load_broker_strategy,
    load_download_period,
)
from planner.download_plan import build_download_plan
from pipelines.binance_runner import _build_month_date_range


config_dir = Path("config")

broker_strategy_df = load_broker_strategy(config_dir)
broker_asset_matrix_df = load_broker_asset_matrix(config_dir)
download_period_df = load_download_period(config_dir)

start_date, end_date = _build_month_date_range("2024-01", "2024-01")

plan = build_download_plan(
    broker_strategy_df=broker_strategy_df,
    broker_asset_matrix_df=broker_asset_matrix_df,
    download_period_df=download_period_df,
    start_date=start_date,
    end_date=end_date,
)

dukascopy_plan = [
    item
    for item in plan
    if item.broker == "dukascopy"
]

[
    (index, item.broker, item.asset, item.broker_symbol, item.window.start_date, item.window.end_date)
    for index, item in enumerate(dukascopy_plan)
]
